In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchcfm.conditional_flow_matching import ConditionalFlowMatcher
from torchdiffeq import odeint
from sklearn.preprocessing import StandardScaler
from scipy.stats import wasserstein_distance
import matplotlib.pyplot as plt
from TrajectoryNet import dataset

In [ ]:
#cell 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

eb_data = dataset.EBData("pcs", max_dim=50)
data = eb_data.data
labels = eb_data.get_times()

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

timepoints = np.unique(labels)

def get_data_by_timepoint(data_scaled, labels):
    data_by_t = {}
    for t in np.unique(labels):
        mask = labels == t
        data_by_t[t] = torch.FloatTensor(data_scaled[mask]).to(device)
    return data_by_t

data_by_t = get_data_by_timepoint(data_scaled, labels)

print(f"Data shape: {data_scaled.shape}")
print(f"Time points: {timepoints}")
print(f"Cells per time point: {[data_by_t[t].shape[0] for t in timepoints]}")

In [ ]:
#cell 3: vector fields

class VectorField(nn.Module):
    """
    Learns the velocity field v(x, t) for flow matching.
    Input: cell state concatenated with time
    Output: velocity (direction + magnitude of movement)
    Architecture follows torchcfm single-cell example:
    3 layers, 64 hidden units, SiLU activations
    """
    def __init__(self, dim, hidden_dim=64):
        super(VectorField, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, dim)
        )
    
    def forward(self, t, x):
        if t.dim() == 0:
            t_vec = t.expand(x.shape[0], 1)
        else:
            t_vec = t.view(-1, 1).expand(x.shape[0], 1)
        inp = torch.cat([x, t_vec], dim=1)
        return self.net(inp)

In [ ]:
#cell 4: training

DIM        = 50
HIDDEN_DIM = 64
EPOCHS     = 500
LR         = 1e-3
N_SAMPLES  = 256

model_fm = VectorField(DIM, HIDDEN_DIM).to(device)
optimizer = torch.optim.Adam(model_fm.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=150, gamma=0.5)

# ConditionalFlowMatcher from torchcfm
FM = ConditionalFlowMatcher(sigma=0.1)

losses = []

print("Training Flow Matching model...")
for epoch in range(EPOCHS):
    model_fm.train()
    optimizer.zero_grad()
    
    epoch_loss = 0.0
    
    # train on all consecutive time point pairs
    for i in range(len(timepoints) - 1):
        t0 = timepoints[i]
        t1 = timepoints[i + 1]
        
        # sample cells from source and target time points
        idx0 = torch.randperm(data_by_t[t0].shape[0])[:N_SAMPLES]
        idx1 = torch.randperm(data_by_t[t1].shape[0])[:N_SAMPLES]
        
        x0 = data_by_t[t0][idx0]
        x1 = data_by_t[t1][idx1]
        
        # normalize time to [0, 1] within this pair
        t_normalized = torch.zeros(N_SAMPLES).to(device)
        
        # get flow matching targets
        t_fm, xt, ut = FM.sample_location_and_conditional_flow(x0, x1)
        
        # scale time back to actual timepoint range
        t_actual = t0 + t_fm * (t1 - t0)
        t_actual = t_actual.to(device)
        
        # predict velocity
        vt = model_fm(t_actual, xt)
        
        # flow matching loss: MSE between predicted and target velocity
        loss = torch.mean((vt - ut) ** 2)
        epoch_loss += loss
    
    epoch_loss.backward()
    torch.nn.utils.clip_grad_norm_(model_fm.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    losses.append(epoch_loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss.item():.6f}")

print("Training complete.")

In [ ]:
#cell 5: plot

plt.figure(figsize=(8, 4))
plt.plot(losses, color='purple')
plt.title('Flow Matching Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.tight_layout()
plt.savefig('../figures/04_fm_training_curve.png', dpi=150)
plt.show()

In [ ]:
#cell 6: generate predictions

def generate_predictions(model, x0, target_times, source_time=0):
    """
    Integrate the learned vector field forward from source_time
    to each target time using the ODE solver.
    """
    model.eval()
    
    # normalize times to match training setup
    t_max = max(target_times)
    t_eval = torch.FloatTensor([source_time] + list(target_times)).to(device)
    
    with torch.no_grad():
        # wrap model for odeint (expects f(t, x) signature)
        def ode_fn(t, x):
            return model(t, x)
        
        z_pred = odeint(ode_fn, x0, t_eval, method='euler', 
                       options={'step_size': 0.1})
    
    return z_pred  # shape: (n_times, n_samples, dim)

In [ ]:
#cell 7

n_traj = 200
idx0 = torch.randperm(data_by_t[0].shape[0])[:n_traj]
x0 = data_by_t[0][idx0]

z_pred = generate_predictions(model_fm, x0, timepoints[1:], source_time=0)
z_pred_np = z_pred.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# trajectories
for i in range(min(50, n_traj)):
    axes[0].plot(z_pred_np[:, i, 0], z_pred_np[:, i, 1],
                alpha=0.2, color='gray', linewidth=0.5)

for t in timepoints:
    obs = data_by_t[t].cpu().numpy()
    axes[0].scatter(obs[:, 0], obs[:, 1], s=5, alpha=0.4, label=f't={t}')

axes[0].set_title('Flow Matching — Trajectories + Observations')
axes[0].set_xlabel('Dim 1')
axes[0].set_ylabel('Dim 2')
axes[0].legend(markerscale=2)

# predicted distributions
colors = plt.cm.viridis(np.linspace(0, 1, len(timepoints)))
for i, t in enumerate(timepoints[1:]):
    axes[1].scatter(z_pred_np[i, :, 0], z_pred_np[i, :, 1],
                   s=10, alpha=0.6, color=colors[i+1], label=f't={t} pred')

axes[1].set_title('Flow Matching — Predicted Distributions')
axes[1].set_xlabel('Dim 1')
axes[1].set_ylabel('Dim 2')
axes[1].legend(markerscale=2)

plt.tight_layout()
plt.savefig('../figures/04_fm_trajectories.png', dpi=150)
plt.show()

In [ ]:
#cell 8

print("Held-out evaluation — Flow Matching in shared 50D PCA space:")
print("-" * 60)

# observations in raw 50D PCA space
obs_by_t_pca = {}
for t in timepoints:
    mask = labels == t
    obs_by_t_pca[t] = data[mask]

n_eval = 500
idx0 = torch.randperm(data_by_t[0].shape[0])[:n_eval]
x0_eval = data_by_t[0][idx0]

z_pred_eval = generate_predictions(model_fm, x0_eval, timepoints[1:], source_time=0)

w_dists = []
for i, t in enumerate(timepoints[1:-1]):
    pred_scaled = z_pred_eval[i].cpu().numpy()
    pred_pca = scaler.inverse_transform(pred_scaled)
    
    obs = obs_by_t_pca[t]
    idx_obs = np.random.choice(len(obs), size=n_eval, replace=False)
    obs_sampled = obs[idx_obs]
    
    w = np.mean([
        wasserstein_distance(pred_pca[:, d], obs_sampled[:, d])
        for d in range(50)
    ])
    w_dists.append(w)
    print(f"  t={t}: Wasserstein = {w:.4f}")

avg_fm = np.mean(w_dists)
print(f"  Average: {avg_fm:.4f}")

print("\n" + "=" * 60)
print("Final comparison — all methods in shared 50D PCA space:")
print(f"  Neural ODE on 50D PCA (direct):      0.2242")
print(f"  Sequential VAE + ODE:                 0.7769")
print(f"  Joint VAE + ODE:                      0.3738")
print(f"  Flow Matching:                        {avg_fm:.4f}")